
## Voice Translator 
#### `Description: Please create a feature that translates English audio into Hindi. The system should listen to the English audio from the user and convert it into Hindi text. If the system does not understand the audio, it should prompt the user to repeat it for better accuracy. This feature should only work with English audio. This translation feature should only be active in a certain time period, like 9.30 PM to 10 PM. Outside the time period the model should say “Taking rest, see you tomorrow!” Guidelines: You should train your own machine learning model. GUI is mandatory for this.`

## Extend the GPU memory

In [13]:
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
 
    except RuntimeError as e:
        print(e)


## Load all the libraries

In [14]:
import numpy as np
import pandas as pd
import tensorflow as tf
import pickle
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences as pad
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Attention, Concatenate
from sklearn.model_selection import train_test_split

## Load the dataset


In [10]:
data = pd.read_csv("eng_hin_dataset.csv")   
data.dropna(inplace=True)
data.drop_duplicates(inplace=True)

## Clean and Preprocess the data set

In [12]:
data["english"] = data["english"].astype(str).str.strip()
data["hindi"] = data["hindi"].astype(str).str.strip()
data = data[(data["english"] != "") & (data["hindi"] != "")]

# Add start & end tokens (ensure this is done before tokenization)
data["hindi"] = data["hindi"].apply(lambda x: "<start> " + x + " <end>")

eng_texts = data["english"].tolist()
hin_texts = data["hindi"].tolist()
print("Samples:", len(eng_texts))
data.head(20)

Samples: 11071


,english,hindi
0,I have to go to sleep.,<start> मुझे सोना है। <end>
1,Muiriel is 20 now.,<start> म्यूरियल अब बीस साल की हो गई है। <end>
2,Muiriel is 20 now.,<start> म्यूरियल अब बीस साल की है। <end>
3,"The password is ""Muiriel"".","<start> कूटशब्द ""Muriel"" है। <end>"
4,"The password is ""Muiriel"".","<start> पासवर्ड ""Muriel"" है। <end>"
5,I will be back soon.,<start> मैं जल्द लौटूंगी। <end>
6,I'm at a loss for words.,<start> मैं तो लाजवाब हो गयी हूँ। <end>
7,This is never going to end.,<start> यह तो कभी खत्म न होगा। <end>
8,I just don't know what to say.,<start> मुझे नहीं पता मैं क्या कहूँ। <end>
9,That was an evil bunny.,<start> वह ख़रगोश दुष्ट था। <end>


## Create Train and Validation set

In [13]:
from sklearn.model_selection import train_test_split

eng_train, eng_val, hin_train, hin_val = train_test_split(
    eng_texts,
    hin_texts,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Train size:", len(eng_train))
print("Val size:", len(eng_val))


Train size: 8856
Val size: 2215


## Create Tokenizers

In [14]:
eng_tokenizer = Tokenizer(filters="", oov_token="<unk>")
hin_tokenizer = Tokenizer(filters="", oov_token="<unk>")

eng_tokenizer.fit_on_texts(eng_train)   # only train set
hin_tokenizer.fit_on_texts(hin_train)


In [57]:
# English sequences
eng_train_seq = eng_tokenizer.texts_to_sequences(eng_train)
eng_val_seq = eng_tokenizer.texts_to_sequences(eng_val)

# Hindi sequences
hin_train_seq = hin_tokenizer.texts_to_sequences(hin_train)
hin_val_seq = hin_tokenizer.texts_to_sequences(hin_val)

# Padding 
encoder_input_train = pad(eng_train_seq, maxlen=MAX_ENG, padding="post")
encoder_input_val = pad(eng_val_seq, maxlen=MAX_ENG, padding="post")

decoder_input_train = pad(hin_train_seq, maxlen=MAX_HIN, padding="post")
decoder_input_val = pad(hin_val_seq, maxlen=MAX_HIN, padding="post")

# Decoder targets  
decoder_target_train = np.zeros_like(decoder_input_train)
decoder_target_val = np.zeros_like(decoder_input_val)

decoder_target_train[:, :-1] = decoder_input_train[:, 1:]
decoder_target_val[:, :-1] = decoder_input_val[:, 1:]

decoder_target_train[:, -1] = 0
decoder_target_val[:, -1] = 0

In [20]:
print("Train shapes:")
print(encoder_input_train.shape, decoder_input_train.shape, decoder_target_train.shape)

print("Val shapes:")
print(encoder_input_val.shape, decoder_input_val.shape)


Train shapes:
(8856, 53) (8856, 59) (8856, 59)
Val shapes:
(2215, 53) (2215, 59)


##  Build Model

In [4]:
# Initiazlize the parameter values

ENG_VOCAB = 7423      
HIN_VOCAB = 7689      
MAX_ENG = 53          
MAX_HIN = 59          
EMBED_DIM = 300
LATENT_DIM = 256
BATCH_SIZE = 64
EPOCHS = 20           


In [24]:
encoder_inputs = Input(shape=(MAX_ENG,), name="encoder_inputs")
enc_emb = Embedding(ENG_VOCAB, EMBED_DIM, mask_zero=True, name="encoder_embedding")(encoder_inputs)

encoder_lstm = LSTM(LATENT_DIM, return_state=True, return_sequences=True, name="encoder_lstm", dropout=0.3, recurrent_dropout=0.3)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)


decoder_inputs = Input(shape=(MAX_HIN,), name="decoder_inputs")
dec_emb_layer = Embedding(HIN_VOCAB, EMBED_DIM, mask_zero=True, name="decoder_embedding")
dec_emb = dec_emb_layer(decoder_inputs)


decoder_lstm = LSTM(LATENT_DIM, return_sequences=True, return_state=True, name="decoder_lstm", dropout=0.3, recurrent_dropout=0.3)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])


attention_layer = Attention(name="attention_layer", use_scale=True)
attention_result = attention_layer([decoder_outputs, encoder_outputs])


decoder_concat_input = Concatenate(axis=-1)([decoder_outputs, attention_result])

decoder_dense = Dense(HIN_VOCAB, activation="softmax", name="output_dense")
decoder_outputs = decoder_dense(decoder_concat_input)


## Compile and Train the model

In [25]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs, name="seq2seq_model")
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",  metrics=["SparseCategoricalAccuracy"])
model.summary()

Model: "seq2seq_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 encoder_inputs (InputLayer)    [(None, 53)]         0           []                               
                                                                                                  
 decoder_inputs (InputLayer)    [(None, 59)]         0           []                               
                                                                                                  
 encoder_embedding (Embedding)  (None, 53, 300)      2226900     ['encoder_inputs[0][0]']         
                                                                                                  
 decoder_embedding (Embedding)  (None, 59, 300)      2306700     ['decoder_inputs[0][0]']         
                                                                                      

In [99]:
history = model.fit(
    [encoder_input_train, decoder_input_train],
    np.expand_dims(decoder_target_train, -1),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(
        [encoder_input_val, decoder_input_val],
        np.expand_dims(decoder_target_val, -1)
    ),
    shuffle=True
)


Epoch 1/20
139/139 [==============================] - 589s 4s/step - loss: 0.8333 - sparse_categorical_accuracy: 0.2160 - val_loss: 0.7010 - val_sparse_categorical_accuracy: 0.2700
Epoch 2/20
139/139 [==============================] - 461s 3s/step - loss: 0.6704 - sparse_categorical_accuracy: 0.2879 - val_loss: 0.6566 - val_sparse_categorical_accuracy: 0.3104
Epoch 3/20
139/139 [==============================] - 716s 5s/step - loss: 0.6141 - sparse_categorical_accuracy: 0.3286 - val_loss: 0.6188 - val_sparse_categorical_accuracy: 0.3474
Epoch 4/20
139/139 [==============================] - 260s 2s/step - loss: 0.5618 - sparse_categorical_accuracy: 0.3647 - val_loss: 0.5920 - val_sparse_categorical_accuracy: 0.3681
Epoch 5/20
139/139 [==============================] - 264s 2s/step - loss: 0.5128 - sparse_categorical_accuracy: 0.3953 - val_loss: 0.5705 - val_sparse_categorical_accuracy: 0.3897
Epoch 6/20
139/139 [==============================] - 265s 2s/step - loss: 0.4645 - sparse_cate

## Save model and tokenizers


In [100]:
model.save("eng_hin_seq2seq.keras")
with open("tokenizers.pkl", "wb") as f:
    pickle.dump((eng_tokenizer, hin_tokenizer, MAX_ENG, MAX_HIN), f)
print("Saved model and tokenizers.")

Saved model and tokenizers.


## Load the model and tokenizers

In [6]:
model = load_model(
    "eng_hin_seq2seq.keras",
    custom_objects={"Attention": Attention}
)




with open("tokenizers.pkl", "rb") as f:
    eng_tokenizer, hin_tokenizer, MAX_ENG, MAX_HIN = pickle.load(f)


## Build and inference model

In [7]:
enc_inf_input = Input(shape=(MAX_ENG,))
enc_inf_emb = model.get_layer("encoder_embedding")(enc_inf_input)

encoder_outputs, h_enc, c_enc = model.get_layer("encoder_lstm")(enc_inf_emb)


encoder_model = Model(enc_inf_input, [encoder_outputs, h_enc, c_enc])



dec_input = Input(shape=(1,))
dec_h = Input(shape=(LATENT_DIM,))
dec_c = Input(shape=(LATENT_DIM,))
enc_out_inf = Input(shape=(MAX_ENG, LATENT_DIM))

dec_emb = model.get_layer("decoder_embedding")(dec_input)

dec_out, h_new, c_new = model.get_layer("decoder_lstm")(
    dec_emb, initial_state=[dec_h, dec_c]
)

attn_out = model.get_layer("attention_layer")([dec_out, enc_out_inf])

concat = Concatenate(axis=-1)([dec_out, attn_out])

dec_logits = model.get_layer("output_dense")(concat)

decoder_model = Model(
    [dec_input, dec_h, dec_c, enc_out_inf],
    [dec_logits, h_new, c_new]
)



## Create a Translate function

In [16]:
reverse_hin = {v: k for k, v in hin_tokenizer.word_index.items()}
start_idx = hin_tokenizer.word_index["<start>"]
end_idx = hin_tokenizer.word_index["<end>"]

def translate(sentence, max_len=MAX_HIN):
    
    seq = eng_tokenizer.texts_to_sequences([sentence])
    seq = pad(seq, maxlen=MAX_ENG, padding="post")
    
    encoder_outputs, h, c = encoder_model.predict(seq, verbose=0)

    
    target_seq = np.array([[start_idx]])
    output_words = []

    for _ in range(max_len):
        preds, h, c = decoder_model.predict(
            [target_seq, h, c, encoder_outputs], verbose=0
        )

        
        token_id = int(np.argmax(preds[0, -1, :]))

        # Stop at <end>
        if token_id == end_idx:
            break

        
        word = reverse_hin.get(token_id, "<unk>")
        output_words.append(word)

        
        target_seq = np.array([[token_id]])

    return " ".join(output_words)


## Translate few dataset sentences

In [17]:
list = data["english"].sample(10)
list

10913                             Don't cause a commotion.
10062                                       Is this Tom's?
8417                                   It's in your hands.
1559                                         I want money.
9924                                This is a great place.
8819     If you help me learn English, I'll help you le...
9677                              You have a lot to learn.
7607                                          It was easy.
7975                                    I'll go to Greece.
978                                  I meant it as a joke.
Name: english, dtype: object

In [18]:
for i in list:
    print(f"\nStatement is : {i} \nPrediction is : {translate(i)}")


Statement is : Don't cause a commotion. 
Prediction is : हल्ला मत मचाओ।

Statement is : Is this Tom's? 
Prediction is : टॉम का है?

Statement is : It's in your hands. 
Prediction is : आपके हाथों में है।

Statement is : I want money. 
Prediction is : मुझे पैसा चाहिए।

Statement is : This is a great place. 
Prediction is : यह बढ़िया जगह है।

Statement is : If you help me learn English, I'll help you learn Japanese. 
Prediction is : अगर तुम ने मेरी अंग्रेज़ी सीखने में मदद सीखने में मदद करूँगी।

Statement is : You have a lot to learn. 
Prediction is : आपको कुछ ज़्यादा ही तेज़ है।

Statement is : It was easy. 
Prediction is : आसान था।

Statement is : I'll go to Greece. 
Prediction is : मैं यूनान जाऊँगी।

Statement is : I meant it as a joke. 
Prediction is : मैंने तो मज़ाक समझकर बोला था।
